# ARIA: End-to-End Tutorial

**Causal-Aware Reasoning for Materials Discovery**

This tutorial walks through the complete ARIA workflow, from loading a knowledge graph to generating material synthesis reports.

## What is ARIA?

ARIA (**A** causal-awa**R**e **I**ntelligent **A**nalysis) is a framework that helps LLMs reason over Processing-Structure-Property (PSP) pathways for materials discovery. Unlike standard RAG, ARIA determines **when** retrieved evidence is **causally sufficient** for scientific decision-making.

## The Key Problem: Contextual Tunneling

The ARIA paper (KDD 2026) identifies **contextual tunneling** — where models over-anchor on narrow retrieved evidence while suppressing global physical reasoning. This occurs even when all retrieved information is factually accurate:

- **Naive KG+LLM forward score:** 0.337 ± 0.027
- **Baseline LLM (no KG):** 0.340 ± 0.033
- **Interpretability:** Naive KG = 0.350 vs Baseline = 0.877 (a **60.4% drop**)

The solution: **inference-time adaptive gating** — activate evidence only when it forms a complete PSP causal chain.

## Three-Tier Causal Cascade

1. **Tier 1 (DIRECT):** PSP-complete path found in KG → use verified causal mechanisms
2. **Tier 2 (ANALOGICAL):** No complete path → transfer from similar material with physical feasibility checks
3. **Tier 3 (FALLBACK):** No KG match → pure LLM reasoning with uncertainty signaling

In [ ]:
# Cell 2: Imports and configuration
import sys
import json
from pathlib import Path

# Core ARIA imports
from aria import ARIAEngine, load_kg, save_kg
from aria.types import (
    ARIAResult, ReasoningTier, EngineMode,
    CausalTraceStep, PSPRelationship, PSPType
)
from aria.kg.graph_store import kg_stats
from aria.kg.diagnostics import KGDiagnostics
from aria.kg.schema import classify_node_layer, classify_path_layers, psp_layers_covered
from aria.retrieval.path_search import find_psp_paths, extract_mechanisms
from aria.retrieval.completeness import (
    causal_completeness_score, identify_missing_layers,
    infer_required_layers, PSPLayer
)
from aria.materials.constraints import validate_synthesis_conditions

# Configuration: set to True if Ollama is running with qwen2:7b
RUN_LIVE = False

print('ARIA imports successful.')
print(f'Run live queries: {RUN_LIVE}')

In [ ]:
# Cell 3: Load demo KG and print stats
kg = load_kg('data/aria_2d_kg_demo.json')
stats = kg_stats(kg)

print('=' * 60)
print('Demo Knowledge Graph Statistics')
print('=' * 60)
print(f'Nodes:              {stats["num_nodes"]}')
print(f'Edges:              {stats["num_edges"]}')
print(f'Density:            {stats["density"]:.4f}')
print(f'Is DAG:             {stats["is_dag"]}')
print(f'Root nodes:         {stats["num_root_nodes"]}')
print(f'Leaf nodes:         {stats["num_leaf_nodes"]}')
print(f'Intermediate nodes: {stats["num_intermediate_nodes"]}')

In [ ]:
# Cell 4: PSP type distribution
from collections import Counter

psp_types = Counter()
for u, v, d in kg.edges(data=True):
    psp_types[d.get('psp_type', 'unknown')] += 1

print('PSP Type Distribution:')
print('-' * 40)
ptype_labels = {
    'Processing_to_Structure': 'Processing -> Structure (P->S)',
    'Structure_to_Property': 'Structure -> Property (S->P)',
    'Processing_to_Property': 'Processing -> Property (P->P shortcut)',
    'Structure_to_Structure': 'Structure -> Structure (S->S)',
}
for ptype, count in sorted(psp_types.items()):
    label = ptype_labels.get(ptype, ptype)
    print(f'  {label}: {count}')

print(f'\nTotal edges: {sum(psp_types.values())}')
print()
print('Note the significant number of P->P shortcuts. These bypass the')
print('structural mediator and are the key cause of contextual tunneling.')

## 1. Exploring the PSP Knowledge Graph

The demo KG contains ~27 relationships covering MoS2 and WS2. It intentionally includes:

- **Complete P->S->P chains** (e.g., CVD temperature 750C -> crystallinity -> carrier mobility)
- **P->P shortcuts** that bypass structure (e.g., CVD temperature 750C -> carrier mobility directly)
- **Orphan edges** with incomplete causal chains
- **Cross-material analogies** (MoS2 <-> WS2)

The **PSP-Complete criterion** (Definition 1 from the paper) states that a path is PSP-complete if it spans all required PSP layers (Processing -> Structure -> Property).

In [ ]:
# Cell 6: Classify nodes by PSP layer
node_layers = {}
for node in kg.nodes():
    layer = classify_node_layer(node)
    if layer:
        node_layers[node] = layer

from collections import Counter
layer_counts = Counter(node_layers.values())
print('Node Classification by PSP Layer:')
print('-' * 40)
for layer, count in sorted(layer_counts.items()):
    print(f'  {layer}: {count} nodes')
print(f'  Unclassified: {len(kg.nodes()) - len(node_layers)} nodes')

In [ ]:
# Cell 7: Find complete P-S-P chains
print('Complete P-S-P Chains (Tier 1 activation):')
print('=' * 60)

# MoS2: CVD temperature 750C -> carrier mobility
paths = find_psp_paths(kg, ['CVD temperature 750C'], ['carrier mobility'], max_hops=4)
for i, p in enumerate(paths, 1):
    score = causal_completeness_score(kg, [p], 'carrier mobility')
    layers = psp_layers_covered(p, kg)
    print(f'\nPath {i}: {" -> ".join(p)}')
    print(f'  Completeness: {score:.2f}')
    print(f'  PSP layers: {layers}')
    if len(p) == 3:
        print(f'  >>> This is a PSP-COMPLETE path (Tier 1)')
    elif len(p) == 2:
        print(f'  >>> This is a P->P SHORTCUT (contextual tunneling risk)')

In [ ]:
# Cell 8: Find incomplete paths — the contextual tunneling traps
print('Incomplete Paths (Contextual Tunneling Traps):')
print('=' * 60)

# P->P shortcuts that bypass structure
shortcut_paths = find_psp_paths(kg, ['CVD temperature 750C'], ['carrier mobility', 'n-type conductivity'], max_hops=2)
for p in shortcut_paths:
    if len(p) == 2:
        ptype = kg.edges[p[0], p[1]].get('psp_type', 'unknown')
        score = causal_completeness_score(kg, [p], 'carrier mobility')
        missing = identify_missing_layers(kg, [p], 'carrier mobility')
        print(f'\nShortcut: {" -> ".join(p)}')
        print(f'  PSP type: {ptype}')
        print(f'  Completeness: {score:.2f}')
        print(f'  Missing layers: {missing}')
        print(f'  >>> This path bypasses the Structure layer!')
        print(f'  >>> A naive KG approach would use this evidence without completeness checking.')

In [ ]:
# Cell 9: Compare completeness scores across path types
print('Completeness Score Comparison:')
print('=' * 60)
print(f'{"Path":50s} {"Score":>8s} {"Complete?":>10s}')
print('-' * 70)

test_paths = [
    (['CVD temperature 750C', 'crystallinity', 'carrier mobility'], 'carrier mobility'),
]

# Find all paths for CVD -> carrier mobility
all_paths = find_psp_paths(kg, ['CVD temperature 750C'], ['carrier mobility'], max_hops=4)
for p in all_paths:
    score = causal_completeness_score(kg, [p], 'carrier mobility')
    is_complete = 'YES' if score >= 1.0 else 'NO'
    path_str = ' -> '.join(p)
    print(f'{path_str:50s} {score:8.2f} {is_complete:>10s}')

print()
print('Key insight: Only the 3-step path (P->S->P) achieves completeness = 1.0.')
print('The 2-step shortcut (P->P) has completeness = 0.50 — it is missing the Structure layer.')
print('This is the mechanistic basis for contextual tunneling.')

### PSP-Complete Criterion (Definition 1)

A path is **PSP-complete** if it spans all required PSP layers for the query. Formally:

$$C(E, q) = \frac{|L(E) \cap L_{\text{req}}(q)|}{|L_{\text{req}}(q)|}$$

where $L(E)$ is the set of PSP layers covered by evidence $E$ and $L_{\text{req}}(q)$ is the set of layers required by query $q$.

- **PSP-complete paths** (Tier 1): $C(E, q) = 1.0$ — all required layers present
- **Incomplete paths** (contextual tunneling risk): $C(E, q) < 1.0$ — missing layers

ARIA gates evidence activation at the **reasoning stage** based on this criterion, unlike standard RAG which activates all retrieved evidence regardless of completeness.

## 2. Running ARIA in Different Modes

ARIA supports five engine modes:

| Mode | KG? | Literature? | CoT? | Description |
|------|-----|-------------|------|-------------|
| `baseline` | No | No | No | Pure LLM, no KG evidence |
| `naive_kg` | Yes | No | No | Simple KG + LLM concatenation |
| `aria` | Yes | No | No | Three-tier causal cascade (default) |
| `aria_search` | Yes | Yes | No | Three-tier + literature search |
| `aria_full` | Yes | Yes | Yes | Three-tier + literature + CoT |

We focus on **baseline**, **naive_kg**, and **aria** to demonstrate the contextual tunneling effect.

In [ ]:
# Cell 12: Mock results for demo (used when Ollama is unavailable)

def get_mock_results():
    """Pre-computed results for demonstration when Ollama is unavailable."""
    results = {}
    
    # Query 1: Forward prediction - CVD MoS2 carrier mobility
    q1 = 'CVD temperature 750C -> carrier mobility (MoS2)'
    results[(q1, 'baseline')] = {
        'answer': {'predicted_property': 'carrier mobility', 'predicted_value': '~30-45 cm2/Vs', 'explanation': 'Based on general materials knowledge, CVD MoS2 at 750C typically achieves carrier mobility in the 30-45 cm2/Vs range.'},
        'tier': ReasoningTier.FALLBACK, 'confidence': 0.55, 'reasoning_type': 'baseline_fallback',
        'causal_trace': [], 'missing_evidence': ['PSP chain for CVD MoS2 carrier mobility'], 'kg_paths_used': 0, 'mode': 'baseline'
    }
    results[(q1, 'naive_kg')] = {
        'answer': {'predicted_property': 'carrier mobility', 'predicted_value': '~40 cm2/Vs', 'explanation': 'CVD temperature 750C increases carrier mobility. Also, Re doping increases carrier mobility. The evidence suggests moderate improvement in mobility.'},
        'tier': ReasoningTier.DIRECT, 'confidence': 0.70, 'reasoning_type': 'naive_kg_concatenation',
        'causal_trace': [CausalTraceStep(processing='CVD temperature 750C', structure='', property_='carrier mobility', confidence=0.65)],
        'missing_evidence': [], 'kg_paths_used': 3, 'mode': 'naive_kg'
    }
    results[(q1, 'aria')] = {
        'answer': {'predicted_property': 'carrier mobility', 'predicted_value': '~40 cm2/Vs', 'explanation': 'CVD at 750C increases crystallinity (evidence: improved grain size and reduced defect density), which in turn increases carrier mobility by reducing charged impurity scattering.'},
        'tier': ReasoningTier.DIRECT, 'confidence': 0.90, 'reasoning_type': 'direct_path',
        'causal_trace': [CausalTraceStep(processing='CVD temperature 750C', structure='crystallinity', property_='carrier mobility', evidence_text='CVD growth at 750C on SiO2 substrate yields large-grain MoS2 with improved crystallinity', confidence=0.90)],
        'missing_evidence': [], 'kg_paths_used': 1, 'mode': 'aria'
    }
    
    # Query 2: Forward prediction - WS2 photoluminescence
    q2 = 'CVD temperature 900C -> photoluminescence quantum yield (WS2)'
    results[(q2, 'baseline')] = {
        'answer': {'predicted_property': 'photoluminescence quantum yield', 'predicted_value': '~0.1-0.5', 'explanation': 'WS2 monolayers typically achieve PL QY around 0.1-0.5, depending on synthesis quality.'},
        'tier': ReasoningTier.FALLBACK, 'confidence': 0.45, 'reasoning_type': 'baseline_fallback',
        'causal_trace': [], 'missing_evidence': ['PSP chain for WS2 PL QY'], 'kg_paths_used': 0, 'mode': 'baseline'
    }
    results[(q2, 'aria')] = {
        'answer': {'predicted_property': 'photoluminescence quantum yield', 'predicted_value': '~0.8-0.99', 'explanation': 'By analogy to MoS2, CVD at 900C increases crystallinity, which increases PL QY. WS2 monolayers with high crystallinity achieve near-unity QY.'},
        'tier': ReasoningTier.ANALOGICAL, 'confidence': 0.72, 'reasoning_type': 'transfer_learning',
        'causal_trace': [CausalTraceStep(processing='CVD temperature 900C', structure='crystallinity', property_='photoluminescence quantum yield', confidence=0.72)],
        'missing_evidence': ['Direct WS2 crystallinity data'], 'kg_paths_used': 1, 'mode': 'aria'
    }
    
    # Query 3: Forward prediction - PtSe2 topological (Tier 3, outside KG)
    q3 = 'Predict topological properties of PtSe2'
    results[(q3, 'baseline')] = {
        'answer': {'predicted_property': 'topological properties', 'explanation': 'PtSe2 may exhibit topological properties depending on film thickness and strain conditions.'},
        'tier': ReasoningTier.FALLBACK, 'confidence': 0.35, 'reasoning_type': 'baseline_fallback',
        'causal_trace': [], 'missing_evidence': ['No KG coverage for PtSe2'], 'kg_paths_used': 0, 'mode': 'baseline'
    }
    results[(q3, 'aria')] = {
        'answer': {'predicted_property': 'topological properties', 'explanation': 'PtSe2 is outside the current KG coverage. Based on parametric knowledge, PtSe2 may show quantum spin Hall effect in monolayer form.'},
        'tier': ReasoningTier.FALLBACK, 'confidence': 0.35, 'reasoning_type': 'parametric_fallback',
        'causal_trace': [], 'missing_evidence': ['No KG coverage for PtSe2', 'No analogical transfer available'], 'kg_paths_used': 0, 'mode': 'aria'
    }
    
    return results

mock_results = get_mock_results()
print(f'Mock results loaded: {len(mock_results)} entries')
print('Set RUN_LIVE = True and ensure Ollama is running to use live results.')

In [ ]:
# Cell 13: Define demo queries
queries = {
    'q1_forward_tier1': {
        'description': 'Forward prediction: CVD MoS2 carrier mobility (should activate Tier 1)',
        'material': 'MoS2',
        'processing': {'temperature': '750C', 'method': 'CVD'},
        'target_property': 'carrier mobility',
    },
    'q2_forward_tier2': {
        'description': 'Forward prediction: CVD WS2 PL QY (should activate Tier 2)',
        'material': 'WS2',
        'processing': {'temperature': '900C', 'method': 'CVD'},
        'target_property': 'photoluminescence quantum yield',
    },
    'q3_forward_tier3': {
        'description': 'Forward prediction: PtSe2 topological properties (Tier 3 fallback)',
        'material': 'PtSe2',
        'processing': {'method': 'CVD'},
        'target_property': 'topological properties',
    },
}

for name, q in queries.items():
    print(f'{name}: {q["description"]}')

In [ ]:
# Cell 14: Run predictions (mock or live)
import time

results_table = []

if RUN_LIVE:
    for mode_name in ['baseline', 'naive_kg', 'aria']:
        engine = ARIAEngine(kg=kg, model='qwen2:7b', mode=mode_name)
        for name, q in queries.items():
            try:
                result = engine.forward_predict(
                    material=q['material'],
                    processing=q['processing'],
                    target_property=q['target_property']
                )
                results_table.append({
                    'query': name, 'mode': mode_name,
                    'tier': result.tier.name, 'confidence': result.confidence,
                    'reasoning_type': result.reasoning_type,
                    'kg_paths_used': result.kg_paths_used,
                })
            except Exception as e:
                results_table.append({
                    'query': name, 'mode': mode_name,
                    'tier': 'ERROR', 'confidence': 0.0,
                    'reasoning_type': str(e), 'kg_paths_used': 0,
                })
else:
    mock = get_mock_results()
    for name, q in queries.items():
        q_desc = q['description'].split(': ')[1] if ': ' in q['description'] else q['description']
        for mode_name in ['baseline', 'naive_kg', 'aria']:
            key = (q_desc, mode_name)
            if key in mock:
                d = mock[key]
                results_table.append({
                    'query': name, 'mode': mode_name,
                    'tier': d['tier'].name, 'confidence': d['confidence'],
                    'reasoning_type': d['reasoning_type'],
                    'kg_paths_used': d['kg_paths_used'],
                })
            else:
                results_table.append({
                    'query': name, 'mode': mode_name,
                    'tier': 'N/A', 'confidence': 0.0,
                    'reasoning_type': 'no mock data', 'kg_paths_used': 0,
                })

print(f'Generated {len(results_table)} result entries')

In [ ]:
# Cell 15: Comparison table
print(f'{"Query":25s} {"Mode":10s} {"Tier":15s} {"Conf":>6s} {"Reasoning":20s} {"KG Paths":>8s}')
print('-' * 90)
for r in results_table:
    print(f'{r["query"]:25s} {r["mode"]:10s} {r["tier"]:15s} {r["confidence"]:6.2f} {r["reasoning_type"]:20s} {r["kg_paths_used"]:8d}')

print()
print('Key observations:')
print('- ARIA (Tier 1) achieves highest confidence for well-covered queries')
print('- Naive KG may have equal or higher raw confidence but lacks mechanistic explanation')
print('- Baseline has no causal trace or KG evidence')
print('- For Tier 3 (no KG coverage), ARIA gracefully falls back to baseline-like reasoning')

## 3. Contextual Tunneling: Why Naive KG Can Harm Reasoning

**Contextual tunneling** (KDD 2026) is the phenomenon where models over-anchor on narrow retrieved evidence while suppressing global physical reasoning. This occurs **even when all retrieved information is factually correct**.

### The Problem

In naive KG+LLM integration, ALL retrieved evidence is provided to the LLM without assessing whether it forms a **causally complete** chain. The LLM then:

1. Anchors on the shortest, most salient path (the P->P shortcut)
2. Suppresses its own parametric reasoning about the missing structural mediator
3. Produces a prediction that is **factually supported but mechanistically incomplete**

### Quantitative Evidence

- Forward prediction: Naive KG+LLM = **0.337** vs Baseline LLM = **0.340** (naive KG is **worse**)
- Interpretability: Naive KG = **0.350** vs Baseline = **0.877** (a **60.4% drop**)

In [ ]:
# Cell 18: Show what naive_kg retrieves
print('What Naive KG+LLM Retrieves for "CVD temperature 750C -> carrier mobility":')
print('=' * 70)
print()
all_paths = find_psp_paths(kg, ['CVD temperature 750C'], ['carrier mobility'], max_hops=4)
mechanisms = extract_mechanisms(kg, all_paths)

print(f'Found {len(all_paths)} paths:')
for i, p in enumerate(all_paths, 1):
    ptype_list = []
    for j in range(len(p)-1):
        edge_data = kg.edges[p[j], p[j+1]]
        ptype_list.append(edge_data.get('psp_type', '?'))
    score = causal_completeness_score(kg, [p], 'carrier mobility')
    missing = identify_missing_layers(kg, [p], 'carrier mobility')
    print(f'\n  Path {i}: {" -> ".join(p)}')
    print(f'  PSP types: {" -> ".join(ptype_list)}')
    print(f'  Completeness: {score:.2f}  Missing: {missing}')

print(f'\nTotal mechanisms extracted: {len(mechanisms)}')
print()
print('>>> Naive_kg concatenates ALL of these paths into the LLM prompt.')
print('>>> The P->P shortcut (Path 2) is shorter and more salient.')
print('>>> The LLM over-anchors on the shortcut, suppressing the mechanistic explanation.')

In [ ]:
# Cell 19: Completeness analysis
print('Completeness Analysis: Complete Path vs. Shortcut')
print('=' * 60)
print()

complete_path = ['CVD temperature 750C', 'crystallinity', 'carrier mobility']
shortcut_path = ['CVD temperature 750C', 'carrier mobility']

for path_name, path in [('Complete (P->S->P)', complete_path), ('Shortcut (P->P)', shortcut_path)]:
    # Create subgraph for this single path
    path_score = causal_completeness_score(kg, [path], 'carrier mobility')
    missing = identify_missing_layers(kg, [path], 'carrier mobility')
    layers = set()
    for node in path:
        layer = classify_node_layer(node)
        if layer:
            layers.add(layer)
    
    print(f'{path_name}:')
    print(f'  Path: {" -> ".join(path)}')
    print(f'  PSP layers covered: {layers}')
    print(f'  Completeness score: {path_score:.2f}')
    print(f'  Missing layers: {missing}')
    print()

print('The complete path covers ALL required PSP layers (Processing, Structure, Property).')
print('The shortcut is missing the Structure layer — this is mechanistic incompleteness.')
print('ARIA gates on this: only PSP-complete paths (completeness = 1.0) activate Tier 1.')

In [ ]:
# Cell 20: Compare naive_kg vs baseline for the same query
print('Comparison: Naive KG vs Baseline vs ARIA')
print('=' * 60)
print()

mock = get_mock_results()
q = 'CVD temperature 750C -> carrier mobility (MoS2)'

for mode in ['baseline', 'naive_kg', 'aria']:
    d = mock.get((q, mode), {})
    print(f'{mode.upper()}:')
    print(f'  Tier: {d.get("tier", ReasoningTier.FALLBACK).name}')
    print(f'  Confidence: {d.get("confidence", 0.0):.2f}')
    print(f'  Reasoning: {d.get("reasoning_type", "N/A")}')
    if d.get('causal_trace'):
        for step in d['causal_trace']:
            print(f'  Trace: {step.processing} -> {step.structure or "(skipped)"} -> {step.property_}')
    else:
        print(f'  Trace: (none — no KG evidence)')
    if d.get('missing_evidence'):
        print(f'  Missing: {d["missing_evidence"]}')
    print()

print('KEY INSIGHT:')
print('- Naive_kg retrieves both complete and shortcut paths, then concatenates them.')
print('- The LLM over-anchors on the shortcut path (CVD 750C -> carrier mobility),')
print('  suppressing mechanistic reasoning through crystallinity.')
print('- ARIA gates on completeness: only the P->S->P chain activates Tier 1.')
print('- Result: ARIA provides a causally complete, mechanistically grounded prediction.')

### How Contextual Tunneling Happens

When naive_kg concatenates **all** retrieved paths into the prompt, the LLM receives both:

1. The **complete P->S->P chain**: CVD 750C -> crystallinity -> carrier mobility (mechanistically grounded)
2. The **P->P shortcut**: CVD 750C -> carrier mobility (bypasses structure)

The LLM over-commits to the **shorter, more salient shortcut**, which is:
- Factually correct (CVD temperature does correlate with carrier mobility)
- Mechanistically incomplete (missing the crystallinity mediator)
- Causally insufficient (cannot explain WHY or HOW)

This is contextual tunneling: **factually accurate evidence that forms an incomplete causal chain**.

## 4. How ARIA Rescues: Inference-Time Adaptive Gating

ARIA prevents contextual tunneling through **inference-time adaptive gating**:

1. **Tier 1 (DIRECT):** Activates **only** when a PSP-complete path exists ($C(E,q) = 1.0$)
2. **Tier 2 (ANALOGICAL):** Activates when no direct path exists, but similar materials have PSP paths (with physical feasibility checks)
3. **Tier 3 (FALLBACK):** No KG match — pure LLM reasoning with explicit uncertainty signaling

The key insight: **selective evidence activation**, gated by causal completeness rather than retrieval confidence alone.

In [ ]:
# Cell 23: ARIA mode routing demonstration
from aria.reasoning.router import ReasoningRouter
from aria.retrieval.similarity import NodeMatcher

print('ARIA Routing Demonstration')
print('=' * 60)
print()

# Create a router for demonstration
matcher = NodeMatcher()
try:
    matcher.precompute(kg)
    matcher_ready = True
except Exception as e:
    print(f'NodeMatcher precompute skipped (sentence-transformers not available): {e}')
    matcher_ready = False

router = ReasoningRouter(llm_client=None, matcher=matcher if matcher_ready else None)

# Show routing for different queries
test_queries = [
    {'description': 'MoS2 CVD carrier mobility (Tier 1)',
     'keywords': ['CVD', 'temperature', 'carrier', 'mobility'],
     'query': {'material': 'MoS2', 'processing': {'temperature': '750C', 'method': 'CVD'}, 'target_property': 'carrier mobility'}},
]

if matcher_ready:
    for tq in test_queries:
        try:
            decision = router.route_forward(tq['query'], kg, matcher)
            print(f'{tq["description"]}:')
            print(f'  Tier: {decision.tier.name}')
            print(f'  Paths: {len(decision.paths) if decision.paths else 0}')
            print(f'  Similarity: {decision.similarity:.2f}')
            print()
        except Exception as e:
            print(f'{tq["description"]}: Error - {e}')
            print()
else:
    print('Router demonstration requires sentence-transformers.')
    print('Expected routing for the demo queries:')
    print('  CVD MoS2 carrier mobility -> Tier 1 (DIRECT)')
    print('  CVD WS2 PL quantum yield   -> Tier 2 (ANALOGICAL)')
    print('  PtSe2 topological properties -> Tier 3 (FALLBACK)')

In [ ]:
# Cell 24: Display full causal trace for ARIA result
mock = get_mock_results()
q = 'CVD temperature 750C -> carrier mobility (MoS2)'
d = mock[(q, 'aria')]

print('ARIA Causal Trace for MoS2 Carrier Mobility:')
print('=' * 60)
print(f'Tier: {d["tier"].name} ({d["tier"].value})')
print(f'Confidence: {d["confidence"]:.2f}')
print(f'Reasoning: {d["reasoning_type"]}')
print()

if d['causal_trace']:
    print('CAUSAL TRACE (Processing -> Structure -> Property):')
    print('-' * 60)
    for i, step in enumerate(d['causal_trace'], 1):
        proc = step.processing or '(N/A)'
        struct = step.structure or '(skipped)'
        prop = step.property_ or '(N/A)'
        print(f'\n  Step {i}:')
        print(f'    Processing: {proc}  [Processing Layer]')
        print(f'    Structure:  {struct}  [Structure Layer]')
        print(f'    Property:   {prop}  [Property Layer]')
        if step.evidence_text:
            print(f'    Evidence: {step.evidence_text[:80]}...' if len(step.evidence_text) > 80 else f'    Evidence: {step.evidence_text}')
        print(f'    Confidence: {step.confidence:.2f}')
else:
    print('No causal trace (fallback mode)')

print(f'\nKG paths used: {d["kg_paths_used"]}')
print(f'Missing evidence: {d.get("missing_evidence", [])}')
print()
print('ARIA provides a COMPLETE P->S->P causal chain with evidence,')
print('unlike naive_kg which provides a shortcut without the structural mediator.')

In [ ]:
# Cell 25: Tier dispatch breakdown
print('Tier Dispatch Across Demo Queries')
print('=' * 60)
print()

tier_dispatch = {
    'CVD MoS2 carrier mobility': 'Tier 1 (DIRECT) — PSP-complete path found',
    'CVD WS2 PL QY': 'Tier 2 (ANALOGICAL) — Similar material (MoS2) used for transfer',
    'PtSe2 topological': 'Tier 3 (FALLBACK) — No KG coverage for PtSe2',
}

for query, tier in tier_dispatch.items():
    print(f'  {query}:')
    print(f'    {tier}')
    print()

print('Paper findings (KDD 2026):')
print('  Tier 1 activates 62.5% of forward queries')
print('  Tier 1 activates 0% of inverse queries (due to CKG topology asymmetry)')
print('  Tier 2 activates 12.5% of forward, 20% of inverse queries')
print('  Tier 3 activates 25% of forward, 80% of inverse queries')

### ARIA Performance Gains

From the KDD 2026 paper:

| System | Forward (ID) | Inverse (ID) | Interpretability |
|--------|-------------|-------------|----------------|
| Baseline LLM | 0.340 | 0.326 | 0.877 |
| Naive KG+LLM | 0.337 | 0.285 | 0.350 |
| ARIA-CORE | **0.410** | **0.377** | **0.833** |
| ARIA-FULL | **0.512** | **0.498** | **0.912** |

- ARIA-CORE: +21.6% vs Naive (forward), +53.8% vs Baseline (ARIA-FULL forward)
- Interpretability: Naive KG collapses to 0.350; ARIA recovers to 0.833-0.912

## 5. Performance Evaluation

ARIA provides multiple evaluation dimensions:

- **Causal Coherence**: Does the prediction follow a valid PSP chain?
- **Source Grounding**: Is the prediction grounded in KG evidence?
- **Internal Validity**: Is the prediction self-consistent and sufficient?
- **PSP Consistency**: Does the output contain a complete P->S->P chain?

In [ ]:
# Cell 28: Load benchmark tasks
from pathlib import Path

benchmark_dir = Path('data/benchmarks')
if benchmark_dir.exists():
    forward_tasks = []
    with open(benchmark_dir / 'forward_prediction.jsonl', 'r') as f:
        for line in f:
            if line.strip():
                forward_tasks.append(json.loads(line))
    print(f'Loaded {len(forward_tasks)} forward prediction tasks')
    if forward_tasks:
        print(f'Sample task: {json.dumps(forward_tasks[0], indent=2)[:300]}...')
else:
    print('Benchmark directory not found. Evaluation requires benchmark files.')
    print('Available benchmarks: forward_prediction.jsonl, inverse_design.jsonl, ood_generalization.jsonl')

In [ ]:
# Cell 29: Pre-computed evaluation results
# These results are from the KDD 2026 paper evaluation

evaluation_results = {
    'System': ['Baseline LLM', 'Naive KG+LLM', 'ARIA-CORE', 'ARIA-FULL'],
    'Forward (ID)': [0.340, 0.337, 0.410, 0.512],
    'Inverse (ID)': [0.326, 0.285, 0.377, 0.498],
    'Interpretability': [0.877, 0.350, 0.833, 0.912],
    'Causal Coherence': [0.45, 0.38, 0.72, 0.78],
    'Source Grounding': [0.00, 0.55, 0.68, 0.82],
}

import pandas as pd
df = pd.DataFrame(evaluation_results)
print('ARIA Evaluation Results (from KDD 2026 paper):')
print('=' * 70)
print(df.to_string(index=False))
print()
print('Key takeaways:')
print('1. Naive KG+LLM underperforms Baseline in forward prediction (0.337 < 0.340)')
print('2. Naive KG+LLM interpretability collapses (0.350 vs Baseline 0.877)')
print('3. ARIA-CORE recovers performance (+21.6% forward vs Naive)')
print('4. ARIA-FULL achieves best results (+53.8% forward vs Baseline)')

In [ ]:
# Cell 30: Visualization (if matplotlib available)
try:
    import matplotlib.pyplot as plt
    from aria.visualization.jhu_theme import setup_jhu_colors, get_jhu_color
    setup_jhu_colors()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    systems = ['Baseline', 'Naive KG', 'ARIA-CORE', 'ARIA-FULL']
    colors = [get_jhu_color('Spirit Blue'), get_jhu_color('Orange'), get_jhu_color('Heritage Blue'), get_jhu_color('Homewood Green')]
    
    # Forward prediction
    axes[0].bar(systems, [0.340, 0.337, 0.410, 0.512], color=colors)
    axes[0].set_title('Forward Prediction (ID)')
    axes[0].set_ylabel('Score')
    axes[0].set_ylim(0, 0.7)
    
    # Interpretability
    axes[1].bar(systems, [0.877, 0.350, 0.833, 0.912], color=colors)
    axes[1].set_title('Interpretability')
    axes[1].set_ylabel('Score')
    axes[1].set_ylim(0, 1.1)
    
    # Causal Coherence
    axes[2].bar(systems, [0.45, 0.38, 0.72, 0.78], color=colors)
    axes[2].set_title('Causal Coherence')
    axes[2].set_ylabel('Score')
    axes[2].set_ylim(0, 1.0)
    
    plt.suptitle('ARIA vs Baseline vs Naive KG: Performance Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('aria_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved to aria_comparison.png')
except ImportError:
    print('matplotlib not available. Skipping visualization.')

## 6. Generating a Material Synthesis Report

ARIA's output can be formatted into a human-readable synthesis report that includes:

- Predicted property and confidence
- Reasoning tier used
- Full causal trace with evidence
- Missing evidence (if any)
- Physical validation checks

In [ ]:
# Cell 33: Format ARIAResult into a human-readable report
def format_synthesis_report(result_data: dict) -> str:
    """Format mock result data into a human-readable material synthesis report."""
    lines = []
    lines.append('=' * 70)
    lines.append('ARIA Material Synthesis Report')
    lines.append('=' * 70)
    lines.append('')
    
    # Prediction summary
    lines.append('PREDICTION SUMMARY')
    lines.append('-' * 40)
    answer = result_data.get('answer', {})
    if isinstance(answer, dict):
        for key, value in answer.items():
            lines.append(f'  {key}: {value}')
    lines.append('')
    
    # Reasoning tier
    lines.append('REASONING TIER')
    lines.append('-' * 40)
    tier = result_data.get('tier', ReasoningTier.FALLBACK)
    tier_descriptions = {
        1: 'DIRECT -- PSP-complete causal path found in KG',
        2: 'ANALOGICAL -- Similar material found, transfer reasoning',
        3: 'FALLBACK -- No KG evidence, pure LLM reasoning',
    }
    lines.append(f'  Tier:           {tier.name} (value={tier.value})')
    lines.append(f'  Description:    {tier_descriptions.get(tier.value, "Unknown")}')
    lines.append(f'  Confidence:     {result_data.get("confidence", 0.0):.3f}')
    lines.append(f'  Reasoning type: {result_data.get("reasoning_type", "N/A")}')
    lines.append(f'  Mode:           {result_data.get("mode", "N/A")}')
    lines.append('')
    
    # Causal trace
    lines.append('CAUSAL TRACE')
    lines.append('-' * 40)
    causal_trace = result_data.get('causal_trace', [])
    if causal_trace:
        for i, step in enumerate(causal_trace, 1):
            proc = step.processing or 'N/A'
            struct = step.structure or '(skipped)'
            prop = step.property_ or 'N/A'
            lines.append(f'  Step {i}:')
            lines.append(f'    {proc} (Processing) -> {struct} (Structure) -> {prop} (Property)')
            if step.evidence_text:
                lines.append(f'    Evidence: {step.evidence_text[:100]}' + ('...' if len(step.evidence_text) > 100 else ''))
            lines.append(f'    Confidence: {step.confidence:.2f}')
    else:
        lines.append('  No causal trace (baseline or fallback mode)')
    lines.append('')
    
    # Missing evidence
    lines.append('MISSING EVIDENCE')
    lines.append('-' * 40)
    missing = result_data.get('missing_evidence', [])
    if missing:
        for ev in missing:
            lines.append(f'  - {ev}')
    else:
        lines.append('  None -- all required evidence was found')
    lines.append('')
    lines.append('=' * 70)
    
    return '\n'.join(lines)

# Generate report for the ARIA result on the canonical query
mock = get_mock_results()
q = 'CVD temperature 750C -> carrier mobility (MoS2)'
report = format_synthesis_report(mock[(q, 'aria')])
print(report)

In [ ]:
# Cell 34: Run physical validation checks
conditions = {
    'material': 'MoS2',
    'temperature_C': 750,
    'substrate': 'SiO2',
    'atmosphere': 'H2/Ar',
    'dopant': '',
    'precursor': 'MoO3',
    'method': 'CVD',
}

validation = validate_synthesis_conditions(conditions)

print('Physical Validation of Synthesis Conditions')
print('=' * 50)
print()
for check, passed in validation.items():
    if isinstance(passed, bool):
        status = 'PASS' if passed else 'FAIL'
        print(f'  {check:35s} {status}')
print()
print('The synthesis conditions (MoS2, 750C, SiO2 substrate) are physically')
print('plausible and consistent with the ARIA prediction.')

## 7. Limitations and Caveats

### Performance Varies by Domain and KG Construction Quality

- ARIA-CORE's HC2 metric (stoichiometric parameters) is **0.360**, *below* baseline's **0.390**. For stoichiometric parameters, the KG may not provide sufficient structural mediation.
- The quality of ARIA's reasoning depends critically on the KG's coverage and curation quality.
- Expert-verified edges (confidence > 0.85) substantially outperform extracted edges (confidence < 0.75).

### Inverse Design Bottleneck

- There is a **3.4x reachability asymmetry** between forward and inverse paths in the KG.
- **0% of inverse queries** activate Tier 1 in the evaluation set.
- Inverse design remains primarily a Tier 3 (fallback) task, limiting ARIA's advantage.

### Scope

- ARIA was evaluated on 2D materials (TMDs, graphene, hBN). Generalization to other material classes requires new domain-specific CKGs.
- The tier logic (DIRECT -> ANALOGICAL -> FALLBACK) is domain-agnostic, but the KG content is domain-specific.

### When ARIA Might Not Help

**Sparse KGs:** If the KG has fewer than ~50 edges with low coverage of PSP-complete paths, ARIA's gating will frequently fall to Tier 3 (fallback). In this regime, ARIA degrades gracefully to baseline performance.

**Novel Domains with No Analogical Transfer:** Tier 2 (analogical) requires that similar materials exist in the KG. If the target material has no structural analogs, Tier 2 cannot activate.

**Tier-3 Fallback Performance:** When neither Tier 1 nor Tier 2 activates, ARIA falls back to pure LLM reasoning. The fallback explicitly discloses uncertainty, which is epistemically honest but does not improve prediction accuracy.

**Analogical Transfer Across Phase Boundaries:** Analogical reasoning may fail when materials differ in fundamental ways (e.g., metallic vs. semiconducting). The similarity threshold in ARIA's router helps mitigate this, but it is not foolproof.

### Extending ARIA to New Domains

ARIA's tier logic is **domain-agnostic**. What changes between domains is the Knowledge Graph:

1. **Build a domain-specific CKG:** Create a `causal_relationships` JSON file following the ARIA schema.
2. **Ensure PSP-complete paths:** The most important feature is having complete P->S->P chains, not just P->P shortcuts.
3. **Use the kg-builder skill:** For semi-automated KG construction from literature.
4. **Validate with domain experts:** Have materials scientists verify the causal relationships.

## 8. Next Steps

This tutorial covered the complete ARIA workflow from KG loading to performance evaluation.

### Existing Tutorials

- **Tutorial 01:** Building and exploring PSP knowledge graphs
- **Tutorial 02:** Forward prediction with ARIA
- **Tutorial 03:** Inverse design with ARIA
- **Tutorial 04:** Causal traces and evaluation

### Production Workflows

- **aria-setup:** Configure ARIA for a new domain
- **aria-run:** Run ARIA on custom queries
- **aria-evaluate:** Compute evaluation metrics
- **kg-builder:** Semi-automated KG construction from literature

### Key Takeaways

1. **Contextual tunneling** is the core failure mode: naive KG+LLM over-anchors on incomplete evidence
2. **PSP-completeness gating** at inference time (not retrieval time) is ARIA's key innovation
3. **Three-tier cascade** (DIRECT -> ANALOGICAL -> FALLBACK) ensures appropriate evidence use
4. **ARIA-FULL** achieves +53.8% over baseline on forward prediction (0.512 vs 0.340)
5. **Limitations exist:** ARIA is scoped to 2D materials and struggles with inverse design

In [ ]:
# Cell 39: Quick reference code snippet
print('"""')
print('ARIA Complete Workflow -- Quick Reference')
print('"""')
print()
print('# 1. Load the knowledge graph')
print('from aria import ARIAEngine, load_kg')
print('from aria.retrieval.completeness import causal_completeness_score')
print('from aria.materials.constraints import validate_synthesis_conditions')
print()
print('kg = load_kg("data/aria_2d_kg_demo.json")')
print()
print('# 2. Initialize the engine')
print('engine = ARIAEngine(kg=kg, model="qwen2:7b", mode="aria")')
print()
print('# 3. Make a forward prediction')
print('result = engine.forward_predict(')
print('    material="MoS2",')
print('    processing={"temperature": "750C", "method": "CVD"},')
print('    target_property="carrier mobility",')
print(')')
print()
print('# 4. Inspect the result')
print('print(f"Tier: {result.tier.name}")')
print('print(f"Confidence: {result.confidence:.3f}")')
print('print(f"Answer: {result.answer}")')
print()
print('# 5. Check causal completeness')
print('from aria.retrieval.path_search import find_psp_paths')
print('paths = find_psp_paths(kg, ["CVD", "temperature"], ["mobility"], max_hops=4)')
print('score = causal_completeness_score(kg, paths, "Predict carrier mobility")')
print('print(f"Completeness: {score:.2f}")')
print()
print('# 6. Validate synthesis conditions')
print('conditions = {"material": "MoS2", "temperature_C": 750, "substrate": "SiO2"}')
print('validation = validate_synthesis_conditions(conditions)')
print('print(f"Valid: {validation['overall_valid']}")')